In [1]:
%%pyspark project.spark.compatibility
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType, MapType
from pyspark.sql import functions as F
from pyspark.sql.functions import lit
from pyspark.sql import SparkSession
from functools import reduce


Executing for connection type: SPARK_GLUE, connection name: project.spark.compatibility
Creating Glue session...
Create session for connection: project.spark.compatibility


'Session ae2xwmv6edkt56-54df8359-64e8-4245-9b20-1a71649bd93a has been created.'

<sagemaker_studio_dataengineering_sessions.sagemaker_base_session_manager.common.debugging_utils.SessionInfoTableDisplay object>

Session created for connection: project.spark.compatibility.

Connection: project.spark.compatibility | Run start time: 2026-05-11 16:25:18.263493 | Run duration : 0:01:38.791205s.


In [2]:
%%pyspark project.spark.compatibility
spark = (
    SparkSession.builder
    .appName("audit_trail_services_3")
    .config("spark.executor.memory", "18g")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.shuffle.partitions", "300")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)



Connection: project.spark.compatibility | Run start time: 2026-05-11 16:26:57.060462 | Run duration : 0:00:02.677687s.


## Q16 — Rule-based layer: high-missing-rate features as customer-problem signals

A feature is null when the underlying behaviour didn't happen for a customer
(e.g. `*_days_for_last_3_events` only exists for users with ≥ 3 events;
`*_recency_weighted_intensity` only when there's recency to weight). High null
rate is therefore not noise — it's the prevalence of a specific customer state.

For each feature we compute the **null-rate gap** between fraud and normal:
- `null_rate_gap = f_null_rate − n_null_rate`
- `nonnull_gap   = f_nonnull_rate − n_nonnull_rate  =  −null_rate_gap`

We rank features by `abs_gap = |null_rate_gap|` rather than by a non-null lift
ratio. Lift can be misleading when the normal non-null rate is near zero
(e.g. 0.001 vs 0.0001 is 10× lift but only a 0.0009 percentage-point gap, too
small to support a population-level rule). The percentage-point gap is what
actually translates into precision and recall when the rule is deployed.

**Per-month split**: the analysis below is run independently for each month
(`2024-12`, `2025-01`, `2025-02`) so that month-to-month stability of the
high-null patterns can be inspected. Each month is exposed as its own
variable (`month202412`, `month202501`, `month202502`) so a single month can
be processed in isolation.


In [3]:
%%pyspark project.spark.compatibility
# Load all 8 event families, KEEPING EACH MONTH SEPARATE.
# Output: month_family_dfs[month][family] = (fraud_df, normal_df)
month202412 = "2024-12-01 00:00:00"
month202501 = "2025-01-01 00:00:00"
month202502 = "2025-02-01 00:00:00"
months = [month202412, month202501, month202502]

# Compact label used for output filenames
month_label = {
    month202412: "2024-12",
    month202501: "2025-01",
    month202502: "2025-02",
}

family_paths = {
    "suspension": (
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_account_suspension_eval/version=1.1/",
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_account_suspension/version=1/",
    ),
    "device_change": (
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_device_change_eval/version=1.1/",
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_device_change/version=1/",
    ),
    "api_call": (
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_enstream_api_call_generic_eval/version=1.1/",
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_enstream_api_call_generic/version=1/",
    ),
    "mno_port": (
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_mno_port_eval/version=1.1/",
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_mno_port/version=1/",
    ),
    "phone_number_change": (
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_phone_number_change_eval/version=1.1/",
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_phone_number_change/version=1/",
    ),
    "sim_change": (
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_sim_change_eval/version=1.1/",
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_sim_change/version=1/",
    ),
    "api_call_partner": (
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_enstream_partner_eval/version=phase1-v1.0/",
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_enstream_partner/version=1/",
    ),
    "customer_stability": (
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_stability_eval/version=phase1-v1.0/",
        "s3://enstream-lake-gold-dev/poc/features/feature_set=time_aware_event_based_behavioural_customer_stability/version=1/",
    ),
}

def load_family_one_month(fraud_base, normal_base, month):
    fraud_path  = f"{fraud_base}dt={month}/"
    normal_path = f"{normal_base}dt={month}/"
    fdf = (spark.read.option("header", "true").option("inferSchema", "true").csv(fraud_path))
    ndf = (spark.read.option("header", "true").option("inferSchema", "true").csv(normal_path))
    return fdf, ndf

month_family_dfs = {}
for m in months:
    month_family_dfs[m] = {
        fam: load_family_one_month(fp, np_, m)
        for fam, (fp, np_) in family_paths.items()
    }

for m in months:
    print(f"\n=== month = {m} ===")
    for fam, (fdf, ndf) in month_family_dfs[m].items():
        print(f"  {fam:25s}  fraud={fdf.count():>8,}  normal={ndf.count():>10,}")



=== month = 2024-12-01 00:00:00 ===
  suspension                 fraud=   3,079  normal=13,349,977
  device_change              fraud=   3,079  normal=13,349,977
  api_call                   fraud=   3,079  normal=13,349,977
  mno_port                   fraud=   3,079  normal=13,349,977
  phone_number_change        fraud=   3,079  normal=13,349,977
  sim_change                 fraud=   3,079  normal=13,349,977
  api_call_partner           fraud=   3,079  normal=13,349,977
  customer_stability         fraud=   3,079  normal=13,349,977

=== month = 2025-01-01 00:00:00 ===
  suspension                 fraud=   2,987  normal=13,882,820
  device_change              fraud=   2,987  normal=13,882,820
  api_call                   fraud=   2,987  normal=13,882,820
  mno_port                   fraud=   2,987  normal=13,882,820
  phone_number_change        fraud=   2,987  normal=13,882,820
  sim_change                 fraud=   2,987  normal=13,882,820
  api_call_partner           fraud=   2,987 

In [4]:
%%pyspark project.spark.compatibility
# Helper: per-column null-rate gap fraud vs normal.
def null_gap_table(df_fraud, df_normal, family, exclude=None):
    exclude = set(exclude or [])
    cols = [c for c in df_fraud.columns if c in df_normal.columns and c not in exclude]
    if not cols:
        return None
    f_total = df_fraud.count()
    n_total = df_normal.count()
    f_aggs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in cols]
    n_aggs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in cols]
    f_row = df_fraud.agg(*f_aggs).collect()[0].asDict()
    n_row = df_normal.agg(*n_aggs).collect()[0].asDict()
    records = []
    for c in cols:
        fr  = round(f_row[c] / f_total, 4) if f_total else None
        nr  = round(n_row[c] / n_total, 4) if n_total else None
        if fr is None or nr is None:
            continue
        gap     = round(fr - nr, 4)
        abs_gap = round(abs(gap), 4)
        records.append((family, c, fr, nr, abs_gap))
    schema = ["family", "col", "f_null_rate", "n_null_rate", "abs_gap"]
    return spark.createDataFrame(records, schema=schema)

exclude_cols = ["customer_id", "reference_id", "reference_ts",
                "from_ts", "to_ts", "dt"]



Connection: project.spark.compatibility | Run start time: 2026-05-11 16:32:30.574529 | Run duration : 0:00:03.243561s.


In [5]:
%%pyspark project.spark.compatibility
# Build per-family gap tables PER MONTH, then union into a per-month master.
# Reports null-rate counts at 0.5 / 0.7 / 0.8 / 0.9 thresholds, plus family
# breakdown at >= 0.9 for spot-checking.
month_all_gaps = {}
for m in months:
    print(f"\n=== month = {m} ===")
    gap_tables = []
    for fam, (fdf, ndf) in month_family_dfs[m].items():
        g = null_gap_table(fdf, ndf, fam, exclude=exclude_cols)
        if g is not None:
            gap_tables.append(g)
    enriched = (reduce(lambda a, b: a.unionByName(b), gap_tables)
                .withColumn("max_null_rate",
                            F.greatest(F.col("f_null_rate"), F.col("n_null_rate")))
                .cache())
    total = enriched.count()
    print(f"  total feature rows across all families: {total}")
    print(f"  null-rate threshold counts (max of f_null, n_null):")
    for thr in [0.5, 0.7, 0.8, 0.9]:
        cnt = enriched.filter(F.col("max_null_rate") >= thr).count()
        print(f"    >= {thr}: {cnt}")
    print(f"  family breakdown at >= 0.9:")
    (enriched
        .filter(F.col("max_null_rate") >= 0.9)
        .groupBy("family")
        .count()
        .orderBy(F.col("count").desc())
        .show(truncate=False))
    month_all_gaps[m] = enriched



=== month = 2024-12-01 00:00:00 ===
  total feature rows across all families: 242
  null-rate threshold counts (max of f_null, n_null):
    >= 0.5: 204
    >= 0.7: 200
    >= 0.8: 144
    >= 0.9: 109
  family breakdown at >= 0.9:
+-------------------+-----+
|family             |count|
+-------------------+-----+
|suspension         |36   |
|phone_number_change|36   |
|sim_change         |16   |
|mno_port           |16   |
|device_change      |3    |
|api_call           |2    |
+-------------------+-----+


=== month = 2025-01-01 00:00:00 ===
  total feature rows across all families: 242
  null-rate threshold counts (max of f_null, n_null):
    >= 0.5: 204
    >= 0.7: 199
    >= 0.8: 144
    >= 0.9: 129
  family breakdown at >= 0.9:
+-------------------+-----+
|family             |count|
+-------------------+-----+
|suspension         |36   |
|mno_port           |36   |
|phone_number_change|36   |
|sim_change         |16   |
|device_change      |3    |
|api_call           |2    |
+----

In [6]:
%%pyspark project.spark.compatibility
# Per-month: list features at the > 0.8 and > 0.9 thresholds.
# Output columns: family, col, f_null_rate, n_null_rate, abs_gap, max_null_rate.
OUTPUT_COLS = ["family", "col", "f_null_rate", "n_null_rate", "abs_gap", "max_null_rate"]

for m in months:
    enriched = month_all_gaps[m]
    for thr in [0.8, 0.9]:
        df = (enriched
            .filter(F.col("max_null_rate") > thr)
            .select(*OUTPUT_COLS)
            .orderBy(F.col("max_null_rate").desc(),
                     F.col("abs_gap").desc()))
        cnt = df.count()
        print(f"\n=== month = {m}  |  max_null_rate > {thr}  |  count = {cnt} ===")
        df.show(cnt, truncate=False)



=== month = 2024-12-01 00:00:00  |  max_null_rate > 0.8  |  count = 144 ===
+-------------------+-----------------------------------------------------------+-----------+-----------+-------+-------------+
|family             |col                                                        |f_null_rate|n_null_rate|abs_gap|max_null_rate|
+-------------------+-----------------------------------------------------------+-----------+-----------+-------+-------------+
|mno_port           |cust_mno_port_interval_min                                 |0.9981     |1.0        |0.0019 |1.0          |
|mno_port           |cust_mno_port_interval_mean                                |0.9981     |1.0        |0.0019 |1.0          |
|mno_port           |cust_mno_port_interval_median                              |0.9981     |1.0        |0.0019 |1.0          |
|mno_port           |cust_mno_port_interval_max                                 |0.9981     |1.0        |0.0019 |1.0          |
|phone_number_change|cust_p

## Optional — Excel export (skip if not needed)

Writes 8 `.xlsx` files into `./excel/` next to the notebook. The same
data is already available in-memory via `month_all_gaps` and the
high-null listing cell above, so this block can be skipped if you don't
need the files on disk.


In [7]:
%%pyspark project.spark.compatibility
# Export 8 Excel files to S3:
#   - 3 months × 2 thresholds (>0.8, >0.9) = 6 files
#   - 1 file: 3-month union, dedup by `col`, max_null_rate > 0.9
#   - 1 file: 3-month union, dedup by `col`, max_null_rate > 0.8
import boto3
from io import BytesIO

try:
    import openpyxl  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl", "--quiet"])

S3_BUCKET = "amazon-sagemaker-004772120527-ca-central-1-86674be5620e"
S3_PREFIX = "dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel"
s3 = boto3.client("s3")

def _save(pdf, filename):
    buf = BytesIO()
    pdf.to_excel(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=S3_BUCKET, Key=f"{S3_PREFIX}/{filename}", Body=buf.getvalue())
    print(f"  saved: s3://{S3_BUCKET}/{S3_PREFIX}/{filename}  ({len(pdf)} rows)")

high_null_results = {}

# 1) Per-month, per-threshold (6 files)
for m in months:
    for thr in [0.8, 0.9]:
        pdf = (month_all_gaps[m]
            .filter(F.col("max_null_rate") > thr)
            .select(*OUTPUT_COLS)
            .orderBy(F.col("max_null_rate").desc())
            .toPandas())
        key = f"{month_label[m]}_gt_{thr}"
        high_null_results[key] = pdf
        _save(pdf, f"high_null_{month_label[m]}_gt_{thr}.xlsx")

# 2) Union across 3 months, dedup by `col`, average metrics
union_df = reduce(lambda a, b: a.unionByName(b),
                  [month_all_gaps[m] for m in months])
dedup_df = (union_df
    .groupBy("family", "col")
    .agg(
        F.round(F.avg("f_null_rate"),   4).alias("f_null_rate"),
        F.round(F.avg("n_null_rate"),   4).alias("n_null_rate"),
        F.round(F.avg("abs_gap"),       4).alias("abs_gap"),
        F.round(F.avg("max_null_rate"), 4).alias("max_null_rate"),
    )
    .cache())

for thr in [0.9, 0.8]:
    pdf = (dedup_df
        .filter(F.col("max_null_rate") > thr)
        .select(*OUTPUT_COLS)
        .orderBy(F.col("max_null_rate").desc())
        .toPandas())
    key = f"3months_dedup_gt_{thr}"
    high_null_results[key] = pdf
    _save(pdf, f"high_null_3months_dedup_gt_{thr}.xlsx")

print(f"\nKeys available in high_null_results: {sorted(high_null_results.keys())}")


  saved: s3://amazon-sagemaker-004772120527-ca-central-1-86674be5620e/dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel/high_null_2024-12_gt_0.8.xlsx  (144 rows)
  saved: s3://amazon-sagemaker-004772120527-ca-central-1-86674be5620e/dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel/high_null_2024-12_gt_0.9.xlsx  (109 rows)
  saved: s3://amazon-sagemaker-004772120527-ca-central-1-86674be5620e/dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel/high_null_2025-01_gt_0.8.xlsx  (144 rows)
  saved: s3://amazon-sagemaker-004772120527-ca-central-1-86674be5620e/dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel/high_null_2025-01_gt_0.9.xlsx  (129 rows)
  saved: s3://amazon-sagemaker-004772120527-ca-central-1-86674be5620e/dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel/high_null_2025-02_gt_0.8.xlsx  (144 rows)
  saved: s3://amazon-sagemaker-004772120527-ca-central-1-86674be5620e/dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel/high_null_2025-02_gt_0.9.xlsx  (130 rows

In [8]:
# Note: this cell does NOT use %%pyspark
import os, boto3

S3_BUCKET = "amazon-sagemaker-004772120527-ca-central-1-86674be5620e"
S3_PREFIX = "dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel"
LOCAL_DIR = "excel"

os.makedirs(LOCAL_DIR, exist_ok=True)
s3 = boto3.client("s3")

resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=S3_PREFIX)
for obj in resp.get("Contents", []):
    fname = os.path.basename(obj["Key"])
    if not fname:
        continue
    local_path = os.path.join(LOCAL_DIR, fname)
    s3.download_file(S3_BUCKET, obj["Key"], local_path)
    print(f"  downloaded: {local_path}")

print(f"\ncwd: {os.getcwd()}")
print(f"files: {sorted(os.listdir(LOCAL_DIR))}")

  downloaded: excel/high_null_2024-12_gt_0.8.xlsx
  downloaded: excel/high_null_2024-12_gt_0.9.xlsx
  downloaded: excel/high_null_2025-01_gt_0.8.xlsx
  downloaded: excel/high_null_2025-01_gt_0.9.xlsx
  downloaded: excel/high_null_2025-02_gt_0.8.xlsx
  downloaded: excel/high_null_2025-02_gt_0.9.xlsx
  downloaded: excel/high_null_3months_dedup_gt_0.8.xlsx
  downloaded: excel/high_null_3months_dedup_gt_0.9.xlsx

cwd: /mnt/custom-file-systems/s3/shared/SherryShi/EDA
files: ['.ipynb_checkpoints', 'high_null_2024-12_gt_0.8.xlsx', 'high_null_2024-12_gt_0.9.xlsx', 'high_null_2025-01_gt_0.8.xlsx', 'high_null_2025-01_gt_0.9.xlsx', 'high_null_2025-02_gt_0.8.xlsx', 'high_null_2025-02_gt_0.9.xlsx', 'high_null_3months_dedup_gt_0.8.xlsx', 'high_null_3months_dedup_gt_0.9.xlsx']


## Pearson correlation (cross-family, per month)

Two-step pairwise Pearson correlation across **all 8 families** (no within-family
restriction). Per month, separately. Threshold: keep pairs with
`|pearson| > 0.9`.

- **Step 1** — all high-null-rate features (from `high_null_results[*_gt_0.9]`,
  unioned across families) × all numeric features in the wide table.
- **Step 2** — all numeric features × all numeric features (upper triangle).

Pipeline per month:
1. Inner chain-join the 8 families on `(customer_id, reference_ts)` → wide
   table with all features. Fraud + normal unioned into a single DF.
2. Cast numeric columns to `double`, mean-impute nulls.
3. One call to `pyspark.ml.stat.Correlation` → full pairwise corr matrix.
4. Slice in driver-side pandas for Step 1 / Step 2.

Notes:
- Inner join keeps only `(customer, reference_ts)` rows present in all 8
  families. If feature names collide across families the join will raise an
  ambiguous-column error — rename upstream if that happens.
- Constant / all-null columns give `NaN` correlations and are filtered out.


In [9]:
%%pyspark project.spark.compatibility
# Chain-join 8 families per month on (customer_id, reference_ts), then union
# fraud + normal. Output: month_wide_dfs[m] = single Spark DF with all features.
JOIN_KEYS       = ["customer_id", "reference_ts"]
drop_from_right = [c for c in exclude_cols if c not in JOIN_KEYS]

month_wide_dfs = {}
for m in months:
    label = month_label[m]
    print(f"\n=== {label}  |  chain join ===")
    fams = list(month_family_dfs[m].keys())

    def chain(side_idx):
        base = month_family_dfs[m][fams[0]][side_idx]
        for fam in fams[1:]:
            right = month_family_dfs[m][fam][side_idx].drop(*drop_from_right)
            base = base.join(right, on=JOIN_KEYS, how="inner")
        return base

    fraud_all  = chain(0)
    normal_all = chain(1)
    wide = fraud_all.unionByName(normal_all, allowMissingColumns=True).cache()
    n_rows, n_cols = wide.count(), len(wide.columns)
    print(f"  wide (fraud+normal): rows={n_rows:>10,}  cols={n_cols}")
    month_wide_dfs[m] = wide



=== 2024-12  |  chain join ===
  wide (fraud+normal): rows=13,353,056  cols=247

=== 2025-01  |  chain join ===
  wide (fraud+normal): rows=13,885,807  cols=247

=== 2025-02  |  chain join ===
  wide (fraud+normal): rows=14,189,712  cols=247
Connection: project.spark.compatibility | Run start time: 2026-05-11 16:35:52.720862 | Run duration : 0:06:26.238662s.


In [10]:
%%pyspark project.spark.compatibility
# Per month: cast numeric cols to double, mean-impute nulls, compute the full
# Pearson correlation matrix in one call. Output:
#   month_corr[m] = (cols, pandas.DataFrame matrix)
import pandas as pd
from pyspark.sql.types import NumericType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

CORR_THR = 0.9

def numeric_cols(df, exclude):
    exclude = set(exclude)
    return [f.name for f in df.schema.fields
            if f.name not in exclude and isinstance(f.dataType, NumericType)]

def corr_matrix(df, exclude):
    cols = numeric_cols(df, exclude)
    if len(cols) < 2:
        return None, None
    means = (df.agg(*[F.avg(F.col(c).cast("double")).alias(c) for c in cols])
               .collect()[0].asDict())
    fill_exprs = [
        F.coalesce(F.col(c).cast("double"),
                   F.lit(means[c] if means[c] is not None else 0.0)).alias(c)
        for c in cols
    ]
    df_filled = df.select(*fill_exprs)
    vec = (VectorAssembler(inputCols=cols, outputCol="__feat")
           .transform(df_filled)
           .select("__feat"))
    mat = Correlation.corr(vec, "__feat", "pearson").head()[0].toArray()
    return cols, pd.DataFrame(mat, index=cols, columns=cols)

month_corr = {}
for m in months:
    label = month_label[m]
    print(f"\n=== {label}  |  computing corr matrix ===")
    cols, mat = corr_matrix(month_wide_dfs[m], exclude_cols)
    month_corr[m] = (cols, mat)
    print(f"  numeric_cols = {len(cols) if cols else 0}")



=== 2024-12  |  computing corr matrix ===
  numeric_cols = 229

=== 2025-01  |  computing corr matrix ===
  numeric_cols = 221

=== 2025-02  |  computing corr matrix ===
  numeric_cols = 225
Connection: project.spark.compatibility | Run start time: 2026-05-11 16:42:19.451513 | Run duration : 0:03:43.246998s.


In [11]:
%%pyspark project.spark.compatibility
# Step 1: all high-null-rate features (> 0.9, across all families) x all numeric
# features in the wide table. Uses high_null_results[f"{label}_gt_0.9"].
month_step1 = {}
for m in months:
    label = month_label[m]
    high_pdf = high_null_results[f"{label}_gt_0.9"]
    # all high-null cols across families, plus a lookup back to the family name
    high_fam = dict(zip(high_pdf["col"], high_pdf["family"]))
    cols, mat = month_corr[m]
    cols_set = set(cols)
    high_cols = [c for c in high_pdf["col"].tolist() if c in cols_set]

    rows = []
    for hc in high_cols:
        for oc in cols:
            if oc == hc:
                continue
            r = mat.loc[hc, oc]
            if pd.notna(r) and abs(r) > CORR_THR:
                rows.append((high_fam.get(hc, ""), hc, oc, round(float(r), 4)))

    pdf = pd.DataFrame(rows, columns=["high_null_family", "high_null_col", "other_col", "pearson"])
    if len(pdf):
        pdf = pdf.reindex(pdf["pearson"].abs().sort_values(ascending=False).index).reset_index(drop=True)
    month_step1[m] = pdf
    print(f"\n=== {label}  |  step1 (high-null x all)  |  pairs > {CORR_THR}: {len(pdf)} ===")
    if len(pdf):
        print(pdf.head(20).to_string(index=False))



=== 2024-12  |  step1 (high-null x all)  |  pairs > 0.9: 90 ===
   high_null_family                                   high_null_col                                       other_col  pearson
phone_number_change        cust_phone_number_change_interval_stddev cust_phone_number_change_days_for_last_3_events   1.0000
phone_number_change           cust_phone_number_change_interval_min        cust_phone_number_change_interval_median   1.0000
           mno_port                      cust_mno_port_interval_min                   cust_mno_port_interval_median   1.0000
phone_number_change    cust_phone_number_change_event_month_entropy          cust_phone_number_change_active_months   1.0000
phone_number_change                 cust_phone_number_change_cnt_1d                 cust_phone_number_change_any_1d   1.0000
phone_number_change        cust_phone_number_change_max_monthly_cnt        cust_phone_number_change_avg_monthly_cnt   1.0000
phone_number_change                 cust_phone_number_change

In [12]:
%%pyspark project.spark.compatibility
# Step 2: all numeric features x all numeric features in the wide table.
# Upper triangle only (no duplicates, no self-pairs).
month_step2 = {}
for m in months:
    label = month_label[m]
    cols, mat = month_corr[m]
    rows = []
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            r = mat.loc[a, b]
            if pd.notna(r) and abs(r) > CORR_THR:
                rows.append((a, b, round(float(r), 4)))

    pdf = pd.DataFrame(rows, columns=["col_a", "col_b", "pearson"])
    if len(pdf):
        pdf = pdf.reindex(pdf["pearson"].abs().sort_values(ascending=False).index).reset_index(drop=True)
    month_step2[m] = pdf
    print(f"\n=== {label}  |  step2 (all x all)  |  pairs > {CORR_THR}: {len(pdf)} ===")
    if len(pdf):
        print(pdf.head(20).to_string(index=False))



=== 2024-12  |  step2 (all x all)  |  pairs > 0.9: 99 ===
                                   col_a                                           col_b  pearson
           cust_mno_port_interval_median                      cust_mno_port_interval_min   1.0000
                    cust_mno_port_cnt_3d                            cust_mno_port_any_3d   1.0000
                    cust_mno_port_cnt_1d                            cust_mno_port_any_1d   1.0000
       customer_enstream_api_call_cnt_3d      customer_enstream_api_call_weighted_cnt_3d   1.0000
       customer_enstream_api_call_cnt_1d      customer_enstream_api_call_weighted_cnt_1d   1.0000
               customer_node_tenure_days                        account_node_tenure_days   1.0000
                account_node_tenure_days                         msisdn_node_tenure_days   1.0000
               customer_node_tenure_days                         msisdn_node_tenure_days   1.0000
      customer_enstream_api_call_cnt_90d     customer_enstr

In [13]:
%%pyspark project.spark.compatibility
# Export 6 Pearson files to S3 (3 months x 2 steps), same prefix as high-null exports.
import boto3
from io import BytesIO

S3_BUCKET = "amazon-sagemaker-004772120527-ca-central-1-86674be5620e"
S3_PREFIX = "dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel"
s3 = boto3.client("s3")

def _save(pdf, filename):
    buf = BytesIO()
    pdf.to_excel(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=S3_BUCKET, Key=f"{S3_PREFIX}/{filename}", Body=buf.getvalue())
    print(f"  saved: s3://{S3_BUCKET}/{S3_PREFIX}/{filename}  ({len(pdf)} rows)")

for m in months:
    label = month_label[m]
    _save(month_step1[m], f"corr_step1_high_null_x_all_{label}_gt_{CORR_THR}.xlsx")
    _save(month_step2[m], f"corr_step2_all_x_all_{label}_gt_{CORR_THR}.xlsx")


  saved: s3://amazon-sagemaker-004772120527-ca-central-1-86674be5620e/dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel/corr_step1_high_null_x_all_2024-12_gt_0.9.xlsx  (90 rows)
  saved: s3://amazon-sagemaker-004772120527-ca-central-1-86674be5620e/dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel/corr_step2_all_x_all_2024-12_gt_0.9.xlsx  (99 rows)
  saved: s3://amazon-sagemaker-004772120527-ca-central-1-86674be5620e/dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel/corr_step1_high_null_x_all_2025-01_gt_0.9.xlsx  (82 rows)
  saved: s3://amazon-sagemaker-004772120527-ca-central-1-86674be5620e/dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel/corr_step2_all_x_all_2025-01_gt_0.9.xlsx  (87 rows)
  saved: s3://amazon-sagemaker-004772120527-ca-central-1-86674be5620e/dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel/corr_step1_high_null_x_all_2025-02_gt_0.9.xlsx  (102 rows)
  saved: s3://amazon-sagemaker-004772120527-ca-central-1-86674be5620e/dzd-6m7qoryrgb21my/ae2xwmv

In [14]:
import os, boto3

S3_BUCKET = "amazon-sagemaker-004772120527-ca-central-1-86674be5620e"
S3_PREFIX = "dzd-6m7qoryrgb21my/ae2xwmv6edkt56/dev/sherry/eda/excel"
LOCAL_DIR = "excel"
os.makedirs(LOCAL_DIR, exist_ok=True)
s3 = boto3.client("s3")

for fname in [
    "corr_step1_high_null_x_all_2024-12_gt_0.9.xlsx",
    "corr_step2_all_x_all_2024-12_gt_0.9.xlsx",
    "corr_step1_high_null_x_all_2025-01_gt_0.9.xlsx",
    "corr_step2_all_x_all_2025-01_gt_0.9.xlsx",
    "corr_step1_high_null_x_all_2025-02_gt_0.9.xlsx",
    "corr_step2_all_x_all_2025-02_gt_0.9.xlsx",
]:
    s3.download_file(S3_BUCKET, f"{S3_PREFIX}/{fname}", f"{LOCAL_DIR}/{fname}")
    print(f"  downloaded: {LOCAL_DIR}/{fname}")

  downloaded: excel/corr_step1_high_null_x_all_2024-12_gt_0.9.xlsx
  downloaded: excel/corr_step2_all_x_all_2024-12_gt_0.9.xlsx
  downloaded: excel/corr_step1_high_null_x_all_2025-01_gt_0.9.xlsx
  downloaded: excel/corr_step2_all_x_all_2025-01_gt_0.9.xlsx
  downloaded: excel/corr_step1_high_null_x_all_2025-02_gt_0.9.xlsx
  downloaded: excel/corr_step2_all_x_all_2025-02_gt_0.9.xlsx
